##### RA5, MANDATORIES EXERCISES
AUTHOR: David Cueli<br>
DATE: 2025-06-03<br>
DESCRIPTION: This notebook contains the mandatory exercises for RA5 step by step<br>

##### KDD PHASES:
- Selección de datos.
- Limpieza de datos.
- Transformation.
- Minería.
- Interpretaciín y evaluación.<br><br>

[V. en]<br>
- Data selection.
- Data cleaning.
- Data transformation.
- Data mining.
- Interpretation and evaluation.

In [171]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

totInitTime = time.time()
qtyDataPreviews = 3

In [172]:
def LBr(pHowMany: int = 1, pComm: str = ""):
  if pComm:
    print(f"+ ")
    print(f"+ {pComm}")

  for i in range(0, pHowMany):
    print(f"+ ")

def Bra(pHowMany: int = 0, pTitle: str = ""):
  LBr(pHowMany)
  if pTitle:
    print(f"+ {pTitle}")
    
  print(f"+ --------------------------------------------------------------------------------------------------------------------------------------")

def Ket(pHowMany: int = 0):
  LBr(pHowMany)
  print(f"+ ======================================================================================================================================")

STEP 1

DATA SELECTION
- Ubicación del Dataset definida en la variable global <<b>PATH_DATASET</b>>
- Importación el Dataset y carga en la variable <<b>df</b>>

[V. en]<br>
- Location path dataset defined in <b>PATH_DATASET</b> global variable
- Import dataset and loading dataset into <b>df</b> variable



In [173]:
PATH_DATASET = "../dataset/amazon_sales.csv"

In [174]:
# 1.
# KDD, Data selection.
# --------------------------------------------------------------------------------------------------------------------------------------

loadDsInitTime = time.time()
# Load the dataset
dfSrc = pd.read_csv(PATH_DATASET)
dfEnd = dfSrc.copy()
# Show the basic structure of the data
Bra(pTitle='Data Preview+Information')
print(f"+ STEP 1: Data Selection completed, loaded from: {PATH_DATASET}")
print(f"+ Shape of dataset: {dfSrc.shape}")
LBr()
dfSrc.info()
LBr()
print(dfSrc.head())
print(f"+ Total Dataset loading execution time: {time.time() - loadDsInitTime:.4f} seconds")
Ket()
LBr()

+ Data Preview+Information
+ --------------------------------------------------------------------------------------------------------------------------------------
+ STEP 1: Data Selection completed, loaded from: ../dataset/amazon_sales.csv
+ Shape of dataset: (1465, 16)
+ 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   product_id           1465 non-null   object
 1   product_name         1465 non-null   object
 2   category             1465 non-null   object
 3   discounted_price     1465 non-null   object
 4   actual_price         1465 non-null   object
 5   discount_percentage  1465 non-null   object
 6   rating               1465 non-null   object
 7   rating_count         1463 non-null   object
 8   about_product        1465 non-null   object
 9   user_id              1465 non-null   object
 10  user_name            1465

In [ ]:
# ============================================================================================================
# DataFrame CleanRegionalFormat
# ------------------------------------------------------------------------------------------------------------
# Limpia y convierte columnas numéricas basadas en el formato regional original (punto para decimales, 
# (,) para miles, símbolos de moneda como (₹) o porcentaje (%).
# 
# @param DataFrame pDfIn DataFrame original con los datos de Amazon en bruto.
# @return DataFrame DataFrame con las columnas numéricas parseadas correctamente.
# ------------------------------------------------------------------------------------------------------------
def CleanRegionalFormat(pDfIn):
  initTime = time.time()
  dfRes = pDfIn.copy()
  
  # 1.
  # Limpieza y conversión de tipos (float) para las columnas de Precios
  dfRes['discounted_price'] = dfRes['discounted_price'].astype(str).str.replace(r'[^\d.,]', '', regex=True).str.replace(',', '').astype(float)
  dfRes['actual_price'] = dfRes['actual_price'].astype(str).str.replace(r'[^\d.,]', '', regex=True).str.replace(',', '').astype(float)
  
  # 2.
  # Limpieza y conversión de tipos (float) de Porcentajes
  dfRes['discount_percentage'] = dfRes['discount_percentage'].astype(str).str.replace(r'[^\d.,]', '', regex=True).str.replace(',', '').astype(float) / 100
  
  # 3.
  # Limpieza y conversión de tipos de Columnas de Conteo (compatibles con NaN)
  dfRes['rating'] = pd.to_numeric(dfRes['rating'].astype(str).str.replace(r'[^\d.,]', '', regex=True).str.replace(',', ''), errors='coerce')
  dfRes['rating_count'] = pd.to_numeric(dfRes['rating_count'].astype(str).str.replace(r'[^\d.,]', '', regex=True).str.replace(',', ''), errors='coerce').astype('Int64')
  
  # 4.
  # VISUALIZACIÓN Y COMPROBACIÓN
  LBr()
  Ket()
  Bra(pTitle='Regional Text Purgation Preview (Functional)')
  print(dfRes[['discounted_price', 'actual_price']].dtypes)
  Bra(pTitle='Viewing Original vs Cleaned Price Columns')
  LBr(0, "Original Price Columns:")
  print(pDfIn[['discounted_price', 'actual_price']].head(qtyDataPreviews))
  LBr(0, "Final Price columns:")
  print(dfRes[['discounted_price', 'actual_price']].head(qtyDataPreviews))
  LBr()
  Bra(pTitle='Total execution time (Clean)')
  print(f"{time.time() - initTime:.4f} seconds")
  Ket()
  LBr()
  
  return dfRes  

# ============================================================================================================
# DataFrame ExplodeUserInteractions
# ------------------------------------------------------------------------------------------------------------
# Transforma la granularidad del dataset explotando los IDs y nombres alineados 1 a 1
# - Separar los IDs múltiples de usuarios y reseñas en registros independientes
# [V.en]
# - Split multiple user and review IDs into independent records
#
# @param DataFrame pDfIn DataFrame limpio a nivel de producto.
# @return DataFrame DataFrame expandido con una fila por usuario/reseña síncrona.
# ------------------------------------------------------------------------------------------------------------
def ExplodeUserInteractions(pDfIn):
  initTime = time.time()
  dfRes = pDfIn.copy()
  
  # 1.
  # Convertir las cadenas de texto separadas por comas en listas de Python
  dfRes['user_id'] = dfRes['user_id'].astype(str).str.split(',')
  dfRes['review_id'] = dfRes['review_id'].astype(str).str.split(',')

  # 2.
  # Explotar el DataFrame para que cada elemento de las listas genere una fila independiente
  dfRes = dfRes.explode(['user_id', 'review_id'])

  # 3.
  # Limpiar posibles espacios en blanco sobrantes alrededor de los IDs extraídos
  dfRes['user_id'] = dfRes['user_id'].str.strip()
  dfRes['review_id'] = dfRes['review_id'].str.strip()

  # 4.
  # VISUALIZACIÓN Y COMPROBACIÓN
  LBr()
  Ket()
  Bra(pTitle='Dataset Transformation Granularity Check')
  LBr(0, "New shape of the expanded dataset:")
  print(f"+ Expanded Shape: {dfRes.shape}")
  LBr(0, "Sample of expanded rows for the same product ID:")
  print(dfRes[['id', 'product_id', 'user_id', 'review_id']].head(qtyDataPreviews))
  LBr()
  Bra(pTitle='Total execution time (Explode)')
  print(f"{time.time() - initTime:.4f} seconds")
  Ket()
  LBr()
  
  return dfRes

# ============================================================================================================
# DataFrame AuditReviewsConsistency
# ------------------------------------------------------------------------------------------------------------
# Audita la veracidad del dataset contrastando el conteo de reseñas físicas con rating_count
#
# @method DataFrame 
# @param DataFrame pDfIn DataFrame explotado con la granularidad por interacción.
# @return DataFrame DataFrame agrupado por producto con las métricas de discrepancia calculadas.
# ------------------------------------------------------------------------------------------------------------
def AuditReviewsConsistency(pDfIn):
  initTime = time.time()
  
  # 1.
  # Agrupamos por producto para contar las reseñas explotadas y extraer el rating_count oficial
  # Usamos 'first' para rating_count ya que es un valor estático nativo del producto
  df = pDfIn.groupby('product_id').agg(
    tot_rev=('review_id', 'count'),
    tot_rat=('rating_count', 'first')
  ).reset_index()
  
  # 2.
  # Calculamos la diferencia absoluta y el porcentaje de desviación matemática
  df['abs_diff'] = df['tot_rat'] - df['tot_rev']
  df['percen_dev'] = (df['abs_diff'] / df['tot_rat']) * 100
  
  # 3.
  # VISUALIZACIÓN Y COMPROBACIÓN
  LBr()
  Ket()
  Bra(pTitle='Data Quality & Veracity Audit Results')
  print(f"+ Analized products: {len(df)}")
  
  LBr(0, "Sample of products with greater discrepancy (Potential wrong data):")
  # Ordenamos de mayor a menor diferencia absoluta para detectar anomalías flagrantes
  print(df.sort_values(by='abs_diff', ascending=False).head(qtyDataPreviews * 2))
  LBr()
  
  # Calculamos métricas globales de integridad del dataset
  totReviews = df['tot_rev'].sum()
  totRatings = df['tot_rat'].sum()
  print(f"+ Total Reviews processed: {totReviews}")
  print(f"+ Total Ratings declared in origin: {totRatings}")
  print(f"+ Real Coverage of the dataset: {(totReviews / totRatings) * 100:.2f}%")
  LBr()
  
  Bra(pTitle='Total execution time (Audit)')
  print(f"{time.time() - initTime:.4f} seconds")
  Ket()
  LBr()
  
  return df

# ============================================================================================================
# DataFrame AuditNullValues
# ------------------------------------------------------------------------------------------------------------
# Identifica el volumen y porcentaje de valores nulos en el dataset
#
# @param DataFrame pDfIn DataFrame expandido y procesado regionalmente.
# @return DataFrame Resumen estadístico de las columnas que contienen datos ausentes.
# ------------------------------------------------------------------------------------------------------------
def AuditNullValues(pDfIn):
  initTime = time.time()
  
  # 1.
  # Cuenta de nulos y su peso porcentual por cada columna
  df = pd.DataFrame({
    'null_values': pDfIn.isnull().sum(),
    'null_percen': (pDfIn.isnull().sum() / len(pDfIn)) * 100
  })
  
  # 2.
  # Filtrar y ordenar para mostrar únicamente las variables afectadas
  df = df[df['null_values'] > 0].sort_values(by='null_values', ascending=False)
  
  # 3.
  # VISUALIZACIÓN Y COMPROBACIÓN
  LBr()
  Ket()
  Bra(pTitle='Null Values & Missing Data Integrity Audit')
  if df.empty:
    print(f"+ The Dataset has no null or missing values.")
  else:
    print(f"+ Columns with missing data detected: {len(df)}")
    print(df)
  LBr()
  
  Bra(pTitle='Total execution time (Null Audit)')
  print(f"{time.time() - initTime:.4f} seconds")
  Ket()
  LBr()
  
  return df

# ============================================================================================================
# DataFrame PurgNullValues
# ------------------------------------------------------------------------------------------------------------
# 
# Elimina de forma estricta los registros que contienen valores nulos en columnas métricas
# 
# @method DataFrame PurgNullValues
# @param DataFrame pDfIn DataFrame con valores ausentes detectados.
# @return DataFrame DataFrame saneado sin filas huérfanas en las columnas críticas.
# ------------------------------------------------------------------------------------------------------------
def PurgeNullValues(pDfIn):
  initTime = time.time()
    
  # 1.
  # Eliminamos las filas con nulos en ambas columnas críticas en un solo paso
  df = pDfIn.dropna(subset=['rating', 'rating_count'])
    
  # 2.
  # VISUALIZACIÓN Y COMPROBACIÓN
  LBr()
  Ket()
  Bra(pTitle='Executing Deletion for Missing Metrics')
  print(f"+ Initial records before purge: {len(pDfIn)}")
  print(f"+ Orphaned records removed: {len(pDfIn) - len(df)}")
  print(f"+ Dimensions of final cleaned dataset: {df.shape}")
  LBr()
  Bra(pTitle='Total execution time (Purge)')
  print(f"{time.time() - initTime:.4f} seconds")
  Ket()
  LBr()
  
  return df

#  *
#  * Descripción de lo que hace la función: Analiza la estructura y volumen de registros en la columna category
#  *
#  * @method DataFrame AuditarColumnaCategoria
#  * @param DataFrame pDfInput DataFrame actual.
#  */
def AuditarColumnaCategoria(pDfInput):
    categoryInitTime = time.time()
    
    nullCount = pDfInput['category'].isnull().sum()
    uniqueCount = pDfInput['category'].nunique()
    
    # ---- Salidas por Consola Integradas ----
    Bra(pTitle='Category Column Structural Audit')
    print(f"+ Valores nulos en categoría: {nullCount}")
    print(f"+ Combinaciones de categorías únicas en bruto: {uniqueCount}")
    LBr(0, "Muestra de los primeros registros de la columna en bruto:")
    print(pDfInput['category'].head(qtyDataPreviews))
    LBr()
    
    Bra(pTitle='Total execution time (Category Audit)')
    print(f"{time.time() - categoryInitTime:.4f} seconds")
    Ket()
    LBr()

#  *
#  * Descripción de lo que hace la función: Extrae el nivel raíz de la jerarquía de categorías para facilitar la agrupación
#  *
#  * @method DataFrame ExtraerCategoriaPrincipal
#  * @param DataFrame pDfInput DataFrame con la columna category original.
#  * @return DataFrame DataFrame con la nueva columna categórica simplificada.
#  */
def ExtraerCategoriaPrincipal(pDfInput):
    processInitTime = time.time()
    dfRes = pDfInput.copy()
    
    # Dividimos por el carácter pipe y extraemos el primer elemento (la raíz)
    dfRes['main_category'] = dfRes['category'].astype(str).str.split('|').str[0].str.strip()
    
    # ---- Salidas por Consola Integradas ----
    Bra(pTitle='Category Transformation - Main Category Extraction')
    print(f"+ Volumen de nuevas categorías principales únicas: {dfRes['main_category'].nunique()}")
    LBr(0, "Distribución de productos por categoría raíz:")
    print(dfRes['main_category'].value_counts())
    LBr()
    
    Bra(pTitle='Total execution time (Category Transformation)')
    print(f"{time.time() - processInitTime:.4f} seconds")
    Ket()
    LBr()
    
    return dfRes


STEP 1.<br>
GENERATING AUTONUMERIC ID<br>

Generamos un ID secuencial único para cada registro


In [176]:
# 1. Generating Autonumeric ID
# Generamos un ID secuencial único para cada registro
# Limpiar caracteres no numéricos según el formato regional (India: (.) para decimales, (,) para miles)

# Creamos la columna autonumérica basada en el índice actual del DataFrame
dfEnd['id'] = range(1, len(dfEnd) + 1)
# Reordenamos las columnas para colocar el 'id' al principio de la tabla
columns = ['id'] + [col for col in dfEnd.columns if col != 'id']
dfEnd = dfEnd[columns]


STEP 2.<br>
DATA CLEANING<br>
DATASET FIRST OVERVIEW 

Al observar el resultado de <<b>df.info()</b>>, se detecta el típico problema técnico de tipo de dato, que no es más que las todas las columnas son de tipo (texto), incluso campos que deberían ser numéricos como <i>discounted_price, actual_price, discount_percentage, rating y rating_count</i>.<br>
Además, <i>rating_count</i> tiene dos valores nulos (1463 no nulos frente a 1465 totales).

Y como son muy pocas filas, y soy de la vieja escuela, al abrir el dataset con una aplicación de de escritorio de hojas de cálculo como puede ser Excel o Google Sheet (si andamos por la nube), se ve claramente que el formato regional del data set es de la India (por el dominio .in las URL de los productos y la moneda en las variables de precios). Por esto habrá que hacer limpieza de los datos de columnas que se detallan a continuación:
  - Eliminaremos completamente las comas (,), ya que actúan exclusivamente como separadores de miles.
  - Como el punto ya es el separador decimal correcto en este formato regional, mantendremos el punto (.).
  - Eliminar cualquier carácter que no sea un dígito como el caracter de moneda (₹) o el (%) y el punto (.).
  - Si existen valores problemáticos que no se pueden transformar en número, los convertirmos en valor NaN o nulo

In [177]:
# 2.
# 2.1 Data Cleaning (Formatting Regional Correction)
# Limpiar y convertir las columnas numéricas basadas en el formato regional original, punto para 
# decimales, (,) para miles, símbolos de moneda como (₹) o porcentaje (%)

dfEnd = CleanRegionalFormat(dfEnd)


+ 
+ ======================================================================================================================================
+ Regional Text Purgation Preview (Functional)
+ --------------------------------------------------------------------------------------------------------------------------------------
discounted_price    float64
actual_price        float64
dtype: object
+ Viewing Original vs Cleaned Price Columns
+ --------------------------------------------------------------------------------------------------------------------------------------
+ 
+ Original Price Columns:
  discounted_price actual_price
0             ₹399       ₹1,099
1             ₹199         ₹349
2             ₹199       ₹1,899
+ 
+ Final Price columns:
   discounted_price  actual_price
0             399.0        1099.0
1             199.0         349.0
2             199.0        1899.0
+ 
+ Total execution time (Clean)
+ ----------------------------------------------------------------------

2.2 Null analysis<br>
Detección, cuantificación y estrategia de tratamiento para los valores nulos o ausentes (NaN) en las variables cuantitativas que acabamos de parsear (rating, rating_count, actual_price, discounted_price).<br>
<br>
Al haber aplicado conversiones de tipo forzadas <b>errors='coerce'</b> en el método <b>to_numeric()</b> de <i>Pandas</i> en los pasos anteriores, cualquier carácter corrupto o inesperado en el origen (como celdas vacías o el famoso carácter erróneo | que suele aparecer en la columna rating de este dataset de Amazon) se habrá transformado automáticamente en un NaN limpio. Ahora toca localizarlos.


In [184]:
AuditarColumnaCategoria(dfEnd)

NameError: name 'AuditarColumnaCategoria' is not defined

STEP 3.<br>
DATA TRANSFORMATION (Structuring)

<b>Granularidad</b> y <b>separación</b> de registros múltiples en las columnas de identificaciones.<br>
Al observar los campos <i>user_id y review_id</i>, vemos que contienen largas cadenas de texto con múltiples IDs separados por comas (por ejemplo: <i>abcd123, ab456, asdf789</i>...). Esto ocurre porque se agrupó todas las reseñas de un mismo producto en una única fila.<br><br>

Para poder hacer un análisis real por usuario o por reseña, debemos "explotar" estas listas.<br><br>

3.1 Data Transformation (Granularity Transformation)
Para separar estas cadenas en listas y luego usar el método <b>.explode()</b> de Pandas. Esto duplicará las filas necesarias, manteniendo el ID autonumérico que creamos antes para saber qué registros pertenecían originalmente al mismo producto.


In [178]:
# 3.
# 3.1. Data Transformation (Granularity Transformation)
# Ejecución de la Transformación de Granularidad para Auditoría
# Separar los IDs múltiples de usuarios y reseñas en registros independientes (Granularidad)

dfEnd = ExplodeUserInteractions(dfEnd)


+ 
+ ======================================================================================================================================
+ Dataset Transformation Granularity Check
+ --------------------------------------------------------------------------------------------------------------------------------------
+ 
+ New shape of the expanded dataset:
+ Expanded Shape: (11503, 17)
+ 
+ Sample of expanded rows for the same product ID:
   id  product_id                       user_id       review_id
0   1  B07JW9H4J1  AG3D6O4STAQKAY2UVGEUV46KN35Q  R3HXWT0LRP0NMF
0   1  B07JW9H4J1  AHMY5CWJMMK5BJRBBSNLYT3ONILA  R2AJM3LFTLZHFO
0   1  B07JW9H4J1  AHCTC6ULH4XB6YHDY6PCH2R772LQ    R6AQJGUP6P86
+ 
+ Total execution time (Explode)
+ --------------------------------------------------------------------------------------------------------------------------------------
0.0403 seconds
+ ==============================================================================================================

3.2 Data Transformation (Category Simplification)<br>
Reducir la dimensionalidad de la columna <b>category</b> quedándonos con la raíz


In [ ]:
# STEP 4.3: KDD - Data Transformation (Category Simplification)
# Reducir la dimensionalidad de la variable category quedándonos con la raíz
# ----------------------------------------------------------------------------------------
dfEnd = ExtraerCategoriaPrincipal(dfEnd)

Con esto generas la columna main_category. Ahora sí que sí, cuando entremos en la Fase 4 (Minería), podremos segmentar los precios, los descuentos y las valoraciones por tipo de producto de forma limpia y directa.

STEP 4.<br>
DATA Analyizing

4.1 Data Evaluation (Quality Audit Execution)<br>
Detección, cuantificación y estrategia de tratamiento para los valores nulos o ausentes (NaN) en las variables cuantitativas que acabamos de parsear (rating, rating_count, actual_price, discounted_price).<br>
<br>
Al haber aplicado conversiones de tipo forzadas <b>errors='coerce'</b> en el método <b>to_numeric()</b> de <i>Pandas</i> en los pasos anteriores, cualquier carácter corrupto o inesperado en el origen (como celdas vacías o el famoso carácter erróneo | que suele aparecer en la columna rating de este dataset de Amazon) se habrá transformado automáticamente en un NaN limpio. Ahora toca localizarlos.


In [179]:
# 4.
# 4.1 Data Evaluation (Quality Audit Execution)
# Ejecutar la revisión de consistencia contra las métricas del propio dataset (rating_count vs conteo real 
# de reseñas explotadas)

dfMetrics = AuditReviewsConsistency(dfEnd)


+ 
+ ======================================================================================================================================
+ Data Quality & Veracity Audit Results
+ --------------------------------------------------------------------------------------------------------------------------------------
+ Analized products: 1351
+ 
+ Sample of products with greater discrepancy (Potential wrong data):
     product_id  tot_rev  tot_rat  abs_diff  percen_dev
137  B014I8SSD0        8   426973    426965   99.998126
138  B014I8SX4Y        8   426973    426965   99.998126
356  B07KSMBL2H       16   426973    426957   99.996253
318  B07GQD4K6L        8   363713    363705     99.9978
317  B07GPXXNNG        8   363713    363705     99.9978
232  B071Z8M4KX        8   363711    363703     99.9978
+ 
+ Total Reviews processed: 11503
+ Total Ratings declared in origin: 23802423
+ Real Coverage of the dataset: 0.05%
+ 
+ Total execution time (Audit)
+ -------------------------------------

In [180]:
# 2.
# 2.2 Data Cleaning (Missing Values Detection)
# Localizar qué variables métricas han generado registros huérfanos tras el parseo regional

dfNullMetrics = AuditNullValues(dfEnd)

+ 
+ ======================================================================================================================================
+ Null Values & Missing Data Integrity Audit
+ --------------------------------------------------------------------------------------------------------------------------------------
+ Columns with missing data detected: 2
              null_values  null_percen
rating                  8     0.069547
rating_count            2     0.017387
+ 
+ Total execution time (Null Audit)
+ --------------------------------------------------------------------------------------------------------------------------------------
0.0255 seconds
+ ======================================================================================================================================
+ 


In [181]:
# 4.
# 4.3 Data Cleaning (Null Values Removal)
# Purgar de forma definitiva los registros nulos para consolidar el dataset estadístico

dfEnd = PurgeNullValues(dfEnd)

+ 
+ ======================================================================================================================================
+ Executing Deletion for Missing Metrics
+ --------------------------------------------------------------------------------------------------------------------------------------
+ Initial records before purge: 11503
+ Orphaned records removed: 10
+ Dimensions of final cleaned dataset: (11493, 17)
+ 
+ Total execution time (Purge)
+ --------------------------------------------------------------------------------------------------------------------------------------
0.0079 seconds
+ ======================================================================================================================================
+ 


APPEND<br>

In [182]:
# # STEP 3.0: KDD - Data Transformation (Granularity & Structural Change)
# # - Separar los IDs múltiples de usuarios y reseñas en registros independientes
# # [V.en]
# # - Split multiple user and review IDs into independent records
# # ----------------------------------------------------------------------------------------
# loadTransInitTime = time.time()

# # 1. Convertir las cadenas de texto separadas por comas en listas de Python
# dfEnd['user_id'] = dfEnd['user_id'].astype(str).str.split(',')
# dfEnd['review_id'] = dfEnd['review_id'].astype(str).str.split(',')

# # 2. Explotar el DataFrame para que cada elemento de las listas genere una fila independiente
# dfEnd = dfEnd.explode(['user_id', 'review_id'])

# # 3. Limpiar posibles espacios en blanco sobrantes alrededor de los IDs extraídos
# dfEnd['user_id'] = dfEnd['user_id'].str.strip()
# dfEnd['review_id'] = dfEnd['review_id'].str.strip()

# # 4. VISUALIZACIÓN Y COMPROBACIÓN
# Bra(pTitle='Dataset Transformation Granularity Check')
# LBr(0, "New shape of the expanded dataset:")
# print(f"+ Expanded Shape: {dfEnd.shape}")
# LBr(0, "Sample of expanded rows for the same product ID:")
# print(dfEnd[['id', 'product_id', 'user_id', 'review_id']].head(qtyDataPreviews * 2))
# LBr()
# Bra(pTitle='Transformation execution time')
# print(f"{time.time() - loadTransInitTime:.4f} seconds")
# Ket()
# LBr()

# # STEP 3.1: KDD - Data Transformation (Text Alignment Check)
# # - Comprobar la estructura de los nombres y reseñas tras la explosión de IDs
# # ----------------------------------------------------------------------------------------
# Bra(pTitle='Checking Text Alignment for Exploded Rows')

# # Observamos las mismas primeras filas para ver si 'user_name' y 'review_title' vienen en listas o texto plano
# print(dfEnd[['id', 'user_id', 'user_name', 'review_title']].head(qtyDataPreviews))

# Ket()
# LBr()

# # STEP 3.2: KDD - Data Transformation (Granularity Conflict Analysis)
# # - Analizar la disparidad de elementos entre IDs y contenidos debido a las comas gramaticales
# # ----------------------------------------------------------------------------------------
# Bra(pTitle='Analyzing Split Counts Disparity')

# # Para evitar conflictos con el explode ya hecho, analizamos los datos directamente desde el origen dfSrc
# # Contamos cuántos elementos salen al separar por comas en cada columna crítica
# print("+ Cantidad de elementos detectados por fila (Muestra de las 3 primeras filas):")

# for i in range(3):
#     u_ids = len(str(dfSrc['user_id'].iloc[i]).split(','))
#     u_names = len(str(dfSrc['user_name'].iloc[i]).split(','))
#     r_titles = len(str(dfSrc['review_title'].iloc[i]).split(','))
#     r_contents = len(str(dfSrc['review_content'].iloc[i]).split(','))
    
#     print(f"\nFila {i+1} (Product ID: {dfSrc['product_id'].iloc[i]}):")
#     print(f"  - IDs de Usuarios (Esperado): {u_ids}")
#     print(f"  - Nombres de Usuarios:        {u_names}")
#     print(f"  - Títulos de Reseñas:         {r_titles}")
#     print(f"  - Contenido de Reseñas:       {r_contents}")

# Ket()
# LBr()

In [183]:
totEndTime = time.time()
Bra()
print(f"+ Total execution time: {totEndTime - totInitTime:.4f} seconds")
Ket()

+ --------------------------------------------------------------------------------------------------------------------------------------
+ Total execution time: 0.4794 seconds
+ ======================================================================================================================================
